<a href="https://colab.research.google.com/github/su-sumico/seminar/blob/main/%E3%83%9E%E3%83%AB%E3%83%81%E3%83%A2%E3%83%BC%E3%83%80%E3%83%AB%E3%83%A2%E3%83%87%E3%83%AB_clip.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Hugging Faceのtransformersライブラリを使用し、事前訓練されたマルチモーダルモデル（例: CLIP）を使って画像キャプションを生成

In [ ]:
!pip install transformers torch pillow requests

In [ ]:
from PIL import Image
import requests
import torch
from transformers import CLIPProcessor, CLIPModel

# CLIPモデルとプロセッサをロード
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# 画像のURLを指定（Pixabayの猫の画像）
url = "https://cdn.pixabay.com/photo/2017/11/09/21/41/cat-2934720_1280.jpg"
image = Image.open(requests.get(url, stream=True).raw)

# 日本語のキャプション候補
texts = [
    "猫の写真",
    "犬の写真",
    "人物の写真",
    "風景の写真",
    "美しい風景",
    "忙しい通り",
    "静かなビーチ"
]

# 画像とテキストをCLIPのフォーマットに変換
inputs = processor(text=texts, images=image, return_tensors="pt", padding=True)

# モデルを使って画像とテキストの類似度を計算
with torch.no_grad():
    outputs = model(**inputs)
    logits_per_image = outputs.logits_per_image  # 画像あたりのログ確率
    probs = logits_per_image.softmax(dim=1)  # ソフトマックスを使って確率に変換

# 最も確率が高いキャプションを選択
max_prob_index = probs.argmax().item()
caption = texts[max_prob_index]

print(f"Generated Caption: {caption}")

Generated Caption: 猫の写真
